## Aviso de versión y resultados vigentes

Este notebook contiene celdas y explicaciones de iteraciones anteriores. Para este corte del proyecto excluimos 2020 de todo análisis y resultado; no uses cifras anteriores que aún aparezcan en celdas o comentarios como resultados finales. Los resultados vigentes se recalcularon con los scripts de la raíz del proyecto y se documentan en Procesos_revisado.docx, README.md y los archivos de resultados_eda/, resultados_dask/ y resultados_modelo_v2/.

La base Parquet completa sin 2020 tiene 24.171.385 filas, 61 variables y 678.550.973 bytes. El modelado usa 23.634.669 vuelos etiquetados. Para AWS/Grafana, la carga real no se verificó; no ejecutes celdas de carga con credenciales sin revisar el destino y subir únicamente agregados analíticos.


> **Decisión metodológica:** 2020 se excluye de todo análisis, modelo, evaluación y visualización. El CSV original se conserva intacto. Los resultados guardados en este notebook se generaron antes de aplicar el filtro y son obsoletos; no deben citarse. Reejecute el flujo con `Year <> 2020` antes de usar resultados. Consulte `DECISION_EXCLUSION_2020.md`.

1. ANALISIS EXPLORATORIO DE LOS DATOS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

Combinamos todos las bases de datos en una sola para facilitar la manipulacion y analisis de al informacion

In [ ]:

# Lista con los nombres de tus archivos
archivos = [
    "Combined_Flights_2018.csv",
    "Combined_Flights_2019.csv",
    "Combined_Flights_2021.csv",
    "Combined_Flights_2022.csv",
]

# Leer y concatenar todos en un solo DataFrame
df_list = [pd.read_csv(file) for file in archivos]
df_flights = pd.concat(df_list, ignore_index=True)
df_flights.head(10)  # Mostrar las primeras 10 filas para verificar la carga de datos

identificamos cada una de las columnas presentes en el Dataframe

In [ ]:
df_flights.columns  # Mostrar los nombres de las columnas del DataFrame

Identificamos el tipo de variable por cada columna

In [ ]:
df_flights.info()  # Mostrar información del DataFrame para verificar tipos de datos y valores nulos

identificamos la estructura del Dataframe

In [ ]:
df_flights.shape # Mostrar el número de filas y columnas del DataFrame

Buscamos si hay valores nulos 

In [ ]:
df_flights.isna().sum()  # Contar valores nulos en cada columna

Buscamos si existen registros duplicados

In [ ]:
df_flights.duplicated().sum()  # Contar filas duplicadas en el DataFrame

In [ ]:

cols = ["ArrDel15", "ArrTimeBlk"]

# 1) Cuántos nulos hay y qué porcentaje del total
print(df_flights[cols].isna().sum())
print((df_flights[cols].isna().mean() * 100).round(2))

# 2) ¿Los nulos de ambas columnas son los mismos registros?
mask_arr = df_flights["ArrDel15"].isna()
mask_blk = df_flights["ArrTimeBlk"].isna()
print("Nulos en ambas:", (mask_arr & mask_blk).sum())
print("Solo ArrDel15:", (mask_arr & ~mask_blk).sum())
print("Solo ArrTimeBlk:", (~mask_arr & mask_blk).sum())

# 3) ¿Coinciden con cancelados y desviados?
print(pd.crosstab(
    df_flights["Cancelled"], df_flights["Diverted"],
    values=mask_arr, aggfunc="sum", margins=True
))
print(df_flights.loc[mask_arr, ["Cancelled", "Diverted"]].value_counts(dropna=False))

# 4) Nulos "sin explicación": no cancelado y no desviado
resto = mask_arr & (df_flights["Cancelled"] == 0) & (df_flights["Diverted"] == 0)
print("Nulos sin explicación operativa:", resto.sum())

# 5) Distribución por año (2020 debería destacar si es el caso)
print(df_flights.loc[mask_arr].groupby("Year").size())

Existen columnas que pueden ser optimizadas para reducir el uso de memoria por lo que convertiremos el tipo de dato a `Category`

In [ ]:
for c in ["Airline", "Origin", "Dest", "OriginState", "DestState"]:
    df_flights[c] = df_flights[c].astype("category")
df_flights["FlightDate"] = pd.to_datetime(df_flights["FlightDate"])

In [ ]:
df_flights.info()  # Verificar los tipos de datos después de la conversión

CLASIFICAR LAS VARIABLES

In [ ]:
resumen = pd.DataFrame({
    "tipo": df_flights.dtypes,
    "valores_distintos": df_flights.nunique(),
})
print(resumen)

In [ ]:
pd.set_option("display.max_rows", None)
print(resumen.sort_values("valores_distintos"))